# Lesson 23: Pulse Voltammetry I: Polarography, NPV and Staircase
Sampled current voltammetry showed us how kinetics, thermodynamics and mass transfer convolve in a stepped experiment. It is rarely used today, because it collects a great deal of data we do not need. This lesson covers the experiments that fixed that.

Today's topics:
- Polarography and the dropping mercury electrode
- Normal pulse voltammetry, and choosing the time parameters
- Staircase voltammetry, and why every digital sweep is really one

**Reference:**
- Sections 7.1 to 7.3 in *Electrochemical Methods: Fundamentals and Applications*, Allen J. Bard & Larry R. Faulkner.
---

## Configure Environment

In [ ]:
import sys
if '../Resources' not in sys.path:
    sys.path.append('../Resources')     # so that echem.py can be found

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

import echem                            # the shared toolkit for this course
from echem import R, T, F, f, WIDE      # constants and the two-panel figure size
echem.use_style()                       # course-wide graph formatting

# Experimental settings shared by the figures in this lesson
POTENTIAL = np.linspace(-0.3, 0.3, 151)     # V vs E0'
TIME = np.linspace(0.005, 2, 500)           # s

# 1. Polarography

Much of Chapter 7 in *Electrochemical Methods* by Bard & Faulkner refers to **polarography**. Polarography is simply voltammetry performed with a mercury droplet as a working electrode. The benefits of Hg electrodes are that the liquid state provides a very easy means to clean the surface of the electrode: simply let the drop fall and a new one form. An additional benefit is that it is as close to an *ideally polarizable electrode* as has been discovered (meaning that pure electrochemistry, without considerations of catalysis, can be studied). Several types of mercury electrodes exist: 
- dropping mercury electrode (DME) - A pool of Hg sits atop a column. The liquid Hg continually travels down the column, forming and releasing droplets at the tip.
- Hanging mercury drop electrode (HMDE) - A Hg droplet is formed but restricted such that it is not released during the course of experiment
- static mercury drop electrode (SMDE) - A blend of DME and HMDE: the Hg droplet is contained at the tip of the column and only released when physically struck.

We will not discuss the application of these electrodes in detail. An important difference between Hg electrodes and solid state electrodes is in mass-transport considerations: the **diffusion layer** (concentration profiles) are effectively erased every time a droplet is released. This obviously does not happen at static solid-state electrodes.

<div class="alert alert-block alert-success">
<b>Ideally polarizable electrode</b>: an electrode at which no charge crosses the interface over the potential range of interest, so that every electron supplied goes into charging the double layer rather than into a reaction.
</div>

# 2. Normal Pulse Voltammetry

Normal pulse voltammetry (NPV) is almost identical to that used for sampled current voltammetry. The key difference is that the current will not be monitored and recorded over the entire experiment.

#### NPV Waveform

The voltage waveform adds a new *time* parameter:

- $\large E_i$: Initial potential. A voltage that is typically chosen to be $E_{OC}$, or some other voltage where no Faradaic charge is passed. 
- $\large \Delta E$: Magnitude of voltage pulse.
- $\large t_1$: Period for which the voltage is pulsed.
- $\large t_2$: Period for which the electrode is allowed to rest between voltage pulses. **Must be large enough to re-establish initial equilibrium.**
- $\tau$: The time at which current is measured and recorded. Given as a value relative to the *start of the pulse*.

In [ ]:
@interact(e_i=widgets.FloatText(value=0.0, step=0.01, description=r'$E$ vs. $E_{OC}$'),
          e_pulse=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E$'),
          t1=widgets.FloatText(value=0.1, step=0.1, description=r'$t_1$'),
          t2=widgets.FloatText(value=0.1, step=0.1, description=r'$t_2$'),
          tau=widgets.FloatText(value=0.05, step=0.1, description=r'$\tau$'))
def plot_npv_waveform(e_i, e_pulse, t1, t2, tau):
    '''Normal pulse voltammetry: sample at tau, then rest so the diffusion layer resets.'''
    if min(t1, t2) <= 0 or not 0 < tau <= t1:
        print('Times must be positive, and the sampling time tau must fall inside t1.')
        return
    t, e = echem.pulse_waveform(e_initial=e_i, e_pulse=e_pulse,
                                t_rest=t2, t_pulse=t1, steps=3)

    fig = plt.figure()
    ax = plt.subplot()
    ax.set_xlabel('t /s')
    ax.set_ylabel(r'E / V vs. $E_{OC}$')

    ax.annotate('', fontsize=6, xy=(t2, e_i + e_pulse*0.5), xycoords='data',
                xytext=(t1 + t2, e_i + e_pulse*0.5), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t_1$', (t1/2 + t2, e_i + e_pulse*0.6), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t1 + t2, e_i + e_pulse*0.5), xycoords='data',
                xytext=(t1 + 2*t2, e_i + e_pulse*0.5), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t_2$', (t1 + 1.5*t2, e_i + e_pulse*0.6), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t2/2, e_i), xycoords='data',
                xytext=(t2/2, e_i + e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E$', (t2/8, e_i + e_pulse/2), fontsize=6)

    ax.annotate('', fontsize=6, xy=(2*t2 + 1.5*t1, e_i), xycoords='data',
                xytext=(2*t2 + 1.5*t1, e_i + 2*e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$2\Delta E$', (2*t2 + 1.6*t1, e_i + e_pulse), fontsize=6)
    ax.annotate(r'$E_i$', (0, e_i), fontsize=6)

    # red markers: the instant at which the current is actually recorded
    for step in range(3):
        ax.scatter([t2 + tau + step*(t2 + t1)], [e_i + (1 + step)*e_pulse],
                   color='red', s=10)
    ax.annotate('', fontsize=6, xy=(t2, e_i + 1.2*e_pulse), xycoords='data',
                xytext=(t2 + tau, e_i + 1.2*e_pulse), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='red'))
    ax.annotate(r'$\tau$', (t2 + tau*0.3, e_i + e_pulse*1.3), fontsize=6, color='red')

    ax.plot(t, e, 'blue')

#### Experimental requirements

It is critically important to NPV experiments that all three *time* parameters be chosen appropriately. Success in the experiment requires that the **diffusion layer** is erased following *every potential pulse*. Poor selection of times will lead to heavily distorted and misleading data.

The NPV experiment generates data that looks the same as that for sampled current voltammetry. For the generic reaction:

$
\large O + ne^- \rightleftharpoons R
$

And the condition that:

$
\large C_R/C_T = 1.0
$

(Take care with current conventions)

<div class="alert alert-block alert-info">
<b>Activity:</b> Grab a pen. Before running the cells below, sketch the current you expect to measure during <i>one</i> pulse and the rest period that follows it.
<br>
Mark the instant at which you would sample, and say why you would sample there rather than at the start of the pulse.
</div>

In [ ]:
# The current sampled at time tau after a step to a given potential comes from
# echem.sampled_current, the same function used in Lesson 19.
#
#     echem.sampled_current(potential, t, conc_total, diff_coeff, area, n)
#
# Sign convention: anodic (oxidation) currents are positive.
help(echem.sampled_current)

In [ ]:
@interact(tau_1=widgets.FloatText(step=0.02, value=0.3, description=r'$\tau$'),
          tau_2=widgets.FloatText(step=0.02, value=0.4, description=r'$\tau_2$'),
          conc_total=widgets.FloatText(step=1e-6, value=1e-6, description=r'$C^*$'))
def plot_npv_output(tau_1, tau_2, conc_total):
    '''Sampling every chronoamperogram at one time turns the family into a wave.'''
    if min(tau_1, tau_2) <= 0 or max(tau_1, tau_2) > TIME.max():
        print(f'Sampling times must lie between 0 and {TIME.max()} s.')
        return
    if conc_total <= 0:
        print('The concentration must be greater than zero.')
        return

    fig = plt.figure(figsize=WIDE)
    ax1 = plt.subplot(121)
    ax1.set_title('Chronoamperometry')
    ax2 = plt.subplot(122)
    ax2.set_title('NPV Output')

    for potential in POTENTIAL:
        ax1.plot(TIME, echem.sampled_current(potential, TIME, conc_total)*1000)
    ax1.set_xlabel(r'$t$ /s')
    ax1.set_ylabel(r'$i$ /mA')
    ax1.set_xlim(0, 0.5)
    for tau, label, colour in [(tau_1, r'$\tau$', 'black'), (tau_2, r'$\tau_2$', 'red')]:
        ax1.axvline(tau, linestyle=':', color=colour)
        ax1.annotate(label, (tau + 0.01, 1.2), color=colour, fontsize=6)
    ax1.annotate(r'+E', (0.05, 1.0), fontsize=6)

    for tau in [tau_1, tau_2]:
        ax2.plot(POTENTIAL, 1000*echem.sampled_current(POTENTIAL, tau, conc_total),
                 label=rf'$\tau$ = {tau} s')
    ax2.set_xlim(-0.2, 0.2)
    ax2.set_xlabel(r"$E$ /V vs E$^{0'}$")
    ax2.set_ylabel(r'$i(\tau)$ /mA')
    ax2.legend(fontsize=6, loc=2)

An underlying assumption in the figure above is that the system is electrochemically reversible. In such situations, we see the **Nernst equation**. When electron transfer kinetics begin to interfere, the behavior will begin to resemble that discussed in [Lesson 21](L21-Irreversibility-Voltammetry.ipynb)

#### How long must $t_2$ be?

The cell above says the rest period must be *large enough to re-establish initial equilibrium*, which is easy to write and considerably harder to act on. Let us make it concrete.

A Nernstian surface concentration snaps back the instant the potential returns to $E_i$, so watching the surface tells you nothing: it always looks recovered. The damage is out in the solution. Each pulse leaves a depleted layer roughly $\delta = \sqrt{6Dt_1}$ thick (the thickness from [Lesson 16](L16-Chronoamperometry.ipynb)), and $t_2$ has to let diffusion refill it before the next pulse arrives.

The widget below runs a train of pulses, all of them stepped well past $E^{0'}$ where the current is fully mass-transport limited. On the left is the concentration of $R$ across the diffusion layer at the end of each rest period. On the right is the NPV wave that results, against the ideal wave you would measure if every pulse started from a fully relaxed solution.

In [ ]:
@interact(t_pulse=widgets.FloatText(value=0.05, step=0.01, description=r'$t_1$ /s'),
          t_rest=widgets.FloatText(value=0.05, step=0.05, description=r'$t_2$ /s'),
          diff_coeff=widgets.FloatText(value=1e-5, step=1e-6, description=r'$D$'))
def plot_npv_recovery(t_pulse, t_rest, diff_coeff):
    """Does the rest period really erase the diffusion layer?"""
    if min(t_pulse, t_rest, diff_coeff) <= 0:
        print('All three values must be greater than zero.')
        return

    e_rest = -0.4                                   # far negative of E0': nothing happens here
    pulses = np.linspace(-0.25, 0.25, 41)           # the NPV scan
    e, durations, sampled = echem.pulse_train_program(pulses, t_pulse, t_rest, e_rest)

    fig = plt.figure(figsize=WIDE)
    ax1 = plt.subplot(121)
    ax2 = plt.subplot(122)

    # LEFT: a train of identical pulses, all on the plateau, so that the only
    # thing changing from one to the next is how much R is left nearby.
    plateau = np.full(6, 0.25)
    e_p, dur_p, _ = echem.pulse_train_program(plateau, t_pulse, t_rest, e_rest)
    x = np.linspace(0, echem.diffusion_layer_thickness(diff_coeff, t_pulse + t_rest), 300)
    for k in range(1, 6):
        t_eval = (k + 1)*t_rest + k*t_pulse         # the end of the rest after pulse k
        ax1.plot(x*1e4, echem.step_train_profile(x, e_p, dur_p, t_eval, diff_coeff),
                 color=echem.color(k - 1), label=f'after pulse {k}')
    ax1.axhline(1.0, linestyle=':', color='black')
    ax1.annotate(r'bulk, $C_R = C^*$', (x.max()*1e4*0.3, 1.03), fontsize=6)
    ax1.set_xlabel(r'$x$ /$\mu$m')
    ax1.set_ylabel(r'$C_R(x)\ /\ C^*$ at end of rest')
    ax1.set_ylim(0, 1.15)
    ax1.legend(fontsize=5, loc=4)

    # RIGHT: the wave this train actually produces, against the ideal one.
    tau = np.where(sampled, t_pulse, t_rest)        # sample at the end of each pulse
    wave = echem.step_train_current(e, durations, tau, 1e-6, diff_coeff)[sampled]
    ideal = echem.sampled_current(pulses, t_pulse, 1e-6, diff_coeff)
    ax2.plot(pulses, ideal*1e6, linestyle=':', color='black', label='fully relaxed')
    ax2.plot(pulses, wave*1e6, color=echem.color(0), label=rf'$t_2$ = {t_rest} s')
    ax2.set_xlabel(r"$E$ /V vs $E^{0'}$")
    ax2.set_ylabel(r'$i(\tau)$ /$\mu$A')
    ax2.legend(fontsize=6, loc=2)

    delta = echem.diffusion_layer_thickness(diff_coeff, t_pulse)
    print(f'delta = sqrt(6 D t1) = {delta*1e4:.1f} um'
          f'   |   t2 / t1 = {t_rest/t_pulse:.1f}'
          f'   |   plateau recovered to {100*wave[-1]/ideal[-1]:.0f} % of ideal')

Hopefully you noticed two things.

First, a wave measured with too short a rest does not simply shrink. It **tilts**. Each successive pulse starts from a more depleted solution than the one before it, so the plateau acquires a slope. A limiting current that still has a slope in it is one of the classic artifacts of this course, and the trouble is that it looks exactly like something physically interesting.

Second, the surface concentration is never what fails to recover. Make $t_2$ as short as you like and $C_R(0)$ still returns to $C^*$ the moment the potential goes back to $E_i$; it is the solution a few micrometres out that is still empty. Hold on to this, because it is the reason *"wait until the current returns to zero"* is not a sufficient test. The current goes to zero as soon as the potential does.

<div class="alert alert-block alert-success">
<b>Choosing the rest time</b>: take $t_2 \gtrsim 5\,t_1$. A pulse of length $t_1$ leaves a depleted layer about $\sqrt{6Dt_1}$ thick, and relaxing that layer takes several times longer than digging it. At $t_2 = 5t_1$ the wave comes back within about 7 % of ideal; at $t_2 = t_1$ it is out by a third.
</div>

# 3. Staircase Voltammetry

Staircase voltammetry provides one significant benefit compared to NPV: it is faster! It achieves this by eliminating the rest time, $t_2$.

#### Staircase Waveform

The staircase waveform is a series of unidirectional potential steps with time.

- $n$: number of steps
- $t$: time spent at each potential step
- $E_i$: initial potential; typically chosen to be $E_{OC}$
- $\Delta E$: height of individual potential steps
- $\nu$: the effective scan rate, defined as:

    $
    \large \nu = \frac{\Delta E}{t}
    $

<div class="alert alert-block alert-success">
<b>Effective scan rate</b>: the average rate at which the potential is advanced in a stepped experiment, $\nu = \Delta E / t$. Every digital "sweep" in the rest of this course is really a staircase whose $\Delta E$ and $t$ are small enough that this average is all anyone quotes.
</div>

In [ ]:
@interact(e_i=widgets.FloatText(value=0.0, step=0.01, description=r'$E_i$'),
          e_step=widgets.FloatText(value=0.01, step=0.01, description=r'$\Delta E$'),
          t_step=widgets.FloatText(value=1., step=1., description=r'$t$'),
          n=widgets.BoundedIntText(value=4, min=2, max=12, description=r'$n$'))
def plot_staircase_waveform(e_i, e_step, t_step, n):
    '''Staircase: the same steps as NPV with the rest periods removed.'''
    if t_step <= 0:
        print('The step time must be greater than zero.')
        return
    t, e = echem.staircase_waveform(e_initial=e_i, e_step=e_step,
                                    t_step=t_step, steps=n)

    fig = plt.figure()
    ax = plt.subplot()
    ax.set_xlabel('t /s')
    ax.set_ylabel(r'E / V vs. $E_{OC}$')

    ax.annotate('', fontsize=6, xy=(t_step, e_i + e_step*0.5), xycoords='data',
                xytext=(2*t_step, e_i + e_step*0.5), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$t$', (t_step*1.5, e_i + e_step*0.6), fontsize=6)

    ax.annotate('', fontsize=6, xy=(t_step*0.5, e_i), xycoords='data',
                xytext=(t_step*0.5, e_i + e_step), textcoords='data',
                arrowprops=dict(arrowstyle='<->', color='black'))
    ax.annotate(r'$\Delta E$', (t_step*0.6, e_i + e_step*0.5), fontsize=6)

    ax.plot(t, e, 'blue')

The removal of the rest time from the experiment prevents the elimination of the diffusion layer. The net effect is a convolution of thermodynamics and mass transfer (and potentially kinetics). Once current begins to flow the **Cottrell equation**, or a related form, begins to factor into the results. The experiment can therefore be completed twice as fast, but results in significantly increased complexity in the $i$ vs. $t$ and $i$ vs. $E$ behavior.

Despite this complexity, the technique found use in polarography. When using DME electrodes the diffusion layer is continually erased, allowing the results to approach the behavior observed during sampled current voltammetry or normal pulse voltammetry.

Because they cannot automatically erase the diffusion layer, **static electrodes** are not typically used for staircase voltammetry this way. 

A sample of data acquired using staircase voltammetry on a DME electrode:

<img src='../Resources/images/staircase.png' width=40%>

<div class="alert alert-block alert-danger">
<b>Memory check:</b> from <a href="L16-Chronoamperometry.ipynb">Lesson 16</a>, write down the Cottrell equation without looking it up.
<br><br>
Now use it to answer two questions. As $t \rightarrow \infty$, is $i$ zero, or merely small? And if the answer is <i>small but not zero</i>, what is still flowing at the instant the <b>next</b> staircase step arrives?
</div>

#### Putting a number on the complexity

The paragraph above claims that deleting the rest period brings "significantly increased complexity", which is worth seeing rather than taking on trust. We already have everything needed to check it.

[Lesson 19](L19-SampledCurrent.ipynb) gave us the current sampled at one time after one step. Diffusion is **linear**, which means the response to a train of steps is nothing more than the sum of the responses to each step taken on its own, each one aged by the time since it happened:

$
\large i(t) = nFAC^*\sqrt{\frac{D}{\pi}}\sum_j \frac{x_j - x_{j-1}}{\sqrt{t-T_j}}
$

Where:
- $T_j$: the time at which step $j$ began, s
- $x_j$: the fraction of the couple converted at the surface during step $j$, set by the **Nernst equation**
- $x_0 = 0$: before the experiment begins the surface sits at its bulk composition

For a single step there is one term, aged by $\tau$, and the sum collapses straight back to the sampled current expression of Lesson 19. This is worth noticing: we have not introduced new physics, only permitted ourselves more than one step.

`echem.step_train_current` is those three lines. Both experiments below are the same function call with different step programs, and `echem.pulse_train_program` builds either one from the same arguments: setting $t_2 = 0$ *is* the staircase experiment.

In [ ]:
@interact(t_step=widgets.FloatText(value=0.05, step=0.01, description=r'$t$ /s'),
          e_step=widgets.FloatText(value=0.01, step=0.005, description=r'$\Delta E$ /V'),
          t_rest=widgets.FloatText(value=0.25, step=0.05, description=r'NPV $t_2$ /s'))
def plot_npv_vs_staircase(t_step, e_step, t_rest):
    """Same steps, same sampling time. The only difference is the rest period."""
    if min(t_step, t_rest, e_step) <= 0:
        print('All three values must be greater than zero.')
        return

    e_rest = -0.4
    scan = np.arange(-0.25, 0.25 + e_step, e_step)

    # Two step programmes that differ ONLY in whether a rest is included.
    e_npv, d_npv, s_npv = echem.pulse_train_program(scan, t_step, t_rest, e_rest)
    e_sc, d_sc, s_sc = echem.pulse_train_program(scan, t_step, t_rest=0.0)

    i_npv = echem.step_train_current(e_npv, d_npv, t_step, 1e-6)[s_npv]
    i_sc = echem.step_train_current(e_sc, d_sc, t_step, 1e-6)[s_sc]

    fig = plt.figure(figsize=WIDE)
    ax1 = plt.subplot(121)
    ax1.set_title('Waveforms')
    ax2 = plt.subplot(122)
    ax2.set_title('Sampled response')

    t1, w1 = echem.pulse_waveform(e_initial=e_rest, e_pulse=e_step,
                                  t_rest=t_rest, t_pulse=t_step, steps=4)
    t2, w2 = echem.staircase_waveform(e_initial=e_rest, e_step=e_step,
                                      t_step=t_step, steps=4)
    ax1.plot(t1, w1, color=echem.color(0), label='NPV')
    ax1.plot(t2, w2, color=echem.color(1), label='staircase')
    ax1.set_xlabel(r'$t$ /s')
    ax1.set_ylabel(r"$E$ /V vs $E^{0'}$")
    ax1.legend(fontsize=6, loc=4)

    ax2.plot(scan, i_npv*1e6, color=echem.color(0), label='NPV')
    ax2.plot(scan, i_sc*1e6, color=echem.color(1), label='staircase')
    ax2.set_xlabel(r"$E$ /V vs $E^{0'}$")
    ax2.set_ylabel(r'$i$ /$\mu$A')
    ax2.legend(fontsize=6, loc=2)

    print(f'NPV:       {d_npv.sum():7.2f} s for the scan, plateau {i_npv[-1]*1e6:7.3f} uA')
    print(f'staircase: {d_sc.sum():7.2f} s for the scan, plateau {i_sc[-1]*1e6:7.3f} uA'
          f'   ({d_npv.sum()/d_sc.sum():.0f} times faster, '
          f'{100*i_sc[-1]/i_npv[-1]:.0f} % of the signal)')

Two things fall out, and they pull in opposite directions.

The staircase experiment finishes in a fraction of the time, by roughly the factor $t_2/t$ that you deleted. That is the entire selling point, and it is not a small one.

What you paid for it is on the right-hand panel, and it is more than a smaller signal. The NPV wave climbs to a plateau and stays there, because every pulse begins from the same fully relaxed solution. The staircase response climbs, turns over, and **decays**. Nothing new is happening chemically. The current falls because the electrode is still eating out of the hole dug by all the previous steps, and that hole keeps deepening as the scan proceeds, so the **Cottrell** decay of everything that came before is now sitting underneath everything you measure.

Look hard at that shape before moving on, because you have seen it before and you will see it constantly from here:

<div class="alert alert-block alert-success">
<b>Why voltammograms have peaks</b>: a wave becomes a peak the moment the diffusion layer is no longer reset between measurements. The rising edge is thermodynamics, the falling edge is depletion, and the peak is where the two hand over.
</div>

This is the whole reason a linear sweep gives a peak rather than a plateau. A digital sweep *is* this staircase with a small $\Delta E$ and a small $t$, so the curve you just generated is a linear sweep voltammogram, arrived at without once mentioning a scan rate. [Lesson 25](L25-SweepVoltammetry.ipynb) puts the equations to it, and [Lesson 27](L27-CyclicVoltammetry.ipynb) uses the sloping tail to explain why measuring a peak current in a cyclic voltammogram requires you to choose a baseline at all.

#### Important note
- Although staircase voltammetry is not used with static electrodes that are most common in modern experiments, it should be understood that modern potentiostats are <b>digital</b>. Linear sweep waveforms (coming in [Lesson 25](L25-SweepVoltammetry.ipynb)) are therefore really staircase voltammetry experiments with small $t$ and $\Delta E$ values. The use of very small values allows the **approximation** of an analog sweep.

<div class="alert alert-block alert-info">
<b>Activity:</b> Staircase voltammetry removes the rest period, so the diffusion layer never resets.
<br>
Sketch the concentration profile at the end of the third step for NPV and for staircase, on the same axes. Which experiment will report the larger current, and which will report the more <i>interpretable</i> one?
</div>

- Larger current: ANSWER
- More interpretable: ANSWER
- Because: ANSWER